# 10 — Model Evaluation and Comparison

This notebook performs the final scientific comparison using **saved Phase 3–4 outputs only**.
It does not load trained model objects, rerun a model, alter parameters, or recreate data.

The analysis separates numerical accuracy from stability, interpretability, complexity, and
robustness. Rankings are multi-criterion summaries rather than proof that one method is
universally superior.


## 1. Imports, paths, and output folders


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="notebook", palette="colorblind",
              rc={"figure.dpi": 120, "savefig.dpi": 300})

def locate_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "results" / "hybrid" / "predictions.csv").exists():
            return candidate
    raise FileNotFoundError("Run from the repository root or notebooks directory.")

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def markdown_table(frame: pd.DataFrame) -> str:
    headers = [str(column) for column in frame.columns]
    lines = ["| " + " | ".join(headers) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in frame.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(lines)

PROJECT_ROOT = locate_project_root(Path.cwd())
FINAL_RESULTS = PROJECT_ROOT / "results" / "final"
FINAL_FIGURES = PROJECT_ROOT / "figures" / "final"
for directory in (FINAL_RESULTS, FINAL_FIGURES):
    directory.mkdir(parents=True, exist_ok=True)


## 2. Load the approved metric outputs


In [ ]:
baseline_metrics = pd.read_csv(PROJECT_ROOT / "results" / "baseline_results.csv")
arima_metrics = pd.read_csv(PROJECT_ROOT / "results" / "arima" / "metrics.csv")
rf_metrics = pd.read_csv(PROJECT_ROOT / "results" / "random_forest" / "metrics.csv")
xgb_metrics = pd.read_csv(PROJECT_ROOT / "results" / "xgboost" / "metrics.csv")
hybrid_metrics_raw = pd.read_csv(PROJECT_ROOT / "results" / "hybrid" / "metrics.csv")

baseline_name_map = {
    "Naive": "Naive baseline",
    "Persistence": "Persistence baseline",
    "Historical Mean": "Historical mean baseline",
    "Moving Average (2-year)": "Moving average baseline",
}
baseline = baseline_metrics.rename(columns={"model": "model_name", "split": "dataset_split"})
baseline["model_name"] = baseline["model_name"].map(baseline_name_map)
baseline = baseline[["model_name", "dataset_split", "MAE", "RMSE", "MAPE", "R2"]]

def normalize_metrics(frame, model_name):
    output = frame.rename(columns={"model": "source_model_name", "split": "dataset_split"}).copy()
    output["model_name"] = model_name
    return output[["model_name", "dataset_split", "MAE", "RMSE", "MAPE", "R2"]]

arima = normalize_metrics(arima_metrics, "ARIMA")
random_forest = normalize_metrics(rf_metrics, "Random Forest")
xgboost = normalize_metrics(xgb_metrics, "XGBoost")

# The three saved hybrid combinations are numerically identical in both partitions.
metric_columns = ["MAE", "RMSE", "MAPE", "R2"]
hybrid_spread = hybrid_metrics_raw.groupby("split")[metric_columns].agg(["min", "max"])
if not np.allclose(
    hybrid_spread.xs("min", axis=1, level=1),
    hybrid_spread.xs("max", axis=1, level=1),
    equal_nan=True,
):
    raise ValueError("Hybrid variants differ; a singular Hybrid row cannot be collapsed transparently.")
hybrid_representative = (
    hybrid_metrics_raw.sort_values(["split", "model"])
    .groupby("split", as_index=False).first()
)
hybrid = normalize_metrics(hybrid_representative, "Hybrid")

comparison = pd.concat([baseline, arima, random_forest, xgboost, hybrid], ignore_index=True)
comparison = comparison.sort_values(["dataset_split", "RMSE", "model_name"]).reset_index(drop=True)
assert comparison.groupby("dataset_split")["model_name"].nunique().eq(8).all()
comparison.to_csv(FINAL_RESULTS / "model_comparison_table.csv", index=False, float_format="%.15g")
display(comparison)


## 3. Load prediction-level outputs and verify alignment

Phase 3 saved baseline metrics but did not save baseline prediction rows. Accordingly, overall
metric tables include all eight approaches, while regional, residual-distribution, and
prediction-level analyses include ARIMA, Random Forest, XGBoost, and the full three-component
hybrid—the approaches with approved prediction files. No baseline predictions are reconstructed.


In [ ]:
prediction_paths = {
    "ARIMA": PROJECT_ROOT / "results" / "arima" / "predictions.csv",
    "Random Forest": PROJECT_ROOT / "results" / "random_forest" / "predictions.csv",
    "XGBoost": PROJECT_ROOT / "results" / "xgboost" / "predictions.csv",
}
predictions = []
for model_name, path in prediction_paths.items():
    frame = pd.read_csv(path)
    frame["model_name"] = model_name
    predictions.append(frame)

hybrid_predictions_raw = pd.read_csv(PROJECT_ROOT / "results" / "hybrid" / "predictions.csv")
preferred_hybrid = "ARIMA + Random Forest + XGBoost"
hybrid_predictions = hybrid_predictions_raw.query("model_name == @preferred_hybrid").copy()
hybrid_predictions["model_name"] = "Hybrid"
predictions.append(hybrid_predictions)
all_predictions = pd.concat(predictions, ignore_index=True)

required = ["region", "year", "actual", "predicted", "residual", "absolute_error",
            "squared_error", "model_name", "dataset_split"]
if any(column not in all_predictions for column in required):
    raise ValueError("A required prediction column is missing.")
if all_predictions.duplicated(["model_name", "dataset_split", "year", "region"]).any():
    raise ValueError("Duplicate prediction keys detected.")

reference = all_predictions.query("model_name == 'ARIMA'")[
    ["dataset_split", "year", "region", "actual"]
].sort_values(["dataset_split", "year", "region"]).reset_index(drop=True)
for model_name, frame in all_predictions.groupby("model_name"):
    keys = frame[["dataset_split", "year", "region", "actual"]].sort_values(
        ["dataset_split", "year", "region"]
    ).reset_index(drop=True)
    if not reference.equals(keys):
        raise ValueError(f"{model_name} prediction keys or actual values are misaligned.")
print("Prediction alignment verified for:", sorted(all_predictions["model_name"].unique()))


## 4. Regional performance, error distributions, and stability


In [ ]:
regional = all_predictions.copy()
regional["absolute_percentage_error"] = (
    100 * regional["absolute_error"] / regional["actual"].replace(0, np.nan)
)
regional_performance = regional[[
    "model_name", "dataset_split", "year", "region", "actual", "predicted",
    "residual", "absolute_error", "squared_error", "absolute_percentage_error",
]].rename(columns={"absolute_error": "MAE", "squared_error": "MSE"})
regional_performance["RMSE"] = np.sqrt(regional_performance["MSE"])
regional_performance.to_csv(
    FINAL_RESULTS / "regional_performance_comparison.csv", index=False, float_format="%.15g"
)

test_regional_stability = (
    regional.query("dataset_split == 'test'")
    .groupby("model_name")["absolute_error"]
    .agg(["mean", "std"])
)
test_regional_stability["regional_error_cv"] = (
    test_regional_stability["std"] / test_regional_stability["mean"]
)
metric_wide = comparison.pivot(index="model_name", columns="dataset_split", values="RMSE")
stability = pd.DataFrame(index=metric_wide.index)
stability["validation_RMSE"] = metric_wide["validation"]
stability["test_RMSE"] = metric_wide["test"]
stability["absolute_log_RMSE_ratio"] = np.abs(
    np.log(stability["test_RMSE"] / stability["validation_RMSE"])
)
stability = stability.join(test_regional_stability[["regional_error_cv"]], how="left")
display(stability.sort_values("absolute_log_RMSE_ratio"))


## 5. Transparent multi-criterion ranking

Numerical ranking uses six equally weighted ranks:

1. Test MAE
2. Test RMSE
3. Test MAPE
4. Test R²
5. Validation RMSE
6. Validation-to-test RMSE stability

This avoids selecting on a single metric. Regional consistency is reported where predictions
exist but is not inserted into the all-model score because baseline prediction rows are absent.
Complexity, interpretability, and robustness are disclosed qualitatively and are not disguised
as measured quantities.


In [ ]:
test_table = comparison.query("dataset_split == 'test'").set_index("model_name")
validation_table = comparison.query("dataset_split == 'validation'").set_index("model_name")
ranking = pd.DataFrame(index=test_table.index)
ranking["test_MAE"] = test_table["MAE"]
ranking["test_RMSE"] = test_table["RMSE"]
ranking["test_MAPE"] = test_table["MAPE"]
ranking["test_R2"] = test_table["R2"]
ranking["validation_RMSE"] = validation_table["RMSE"]
ranking["rmse_stability"] = stability["absolute_log_RMSE_ratio"]
ranking["rank_test_MAE"] = ranking["test_MAE"].rank(method="min")
ranking["rank_test_RMSE"] = ranking["test_RMSE"].rank(method="min")
ranking["rank_test_MAPE"] = ranking["test_MAPE"].rank(method="min")
ranking["rank_test_R2"] = ranking["test_R2"].rank(ascending=False, method="min")
ranking["rank_validation_RMSE"] = ranking["validation_RMSE"].rank(method="min")
ranking["rank_stability"] = ranking["rmse_stability"].rank(method="min")
rank_columns = [column for column in ranking if column.startswith("rank_")]
ranking["mean_numeric_rank"] = ranking[rank_columns].mean(axis=1)

qualitative = {
    "Naive baseline": ("Very low", "Very high", "High"),
    "Persistence baseline": ("Very low", "Very high", "High"),
    "Historical mean baseline": ("Very low", "Very high", "High"),
    "Moving average baseline": ("Very low", "Very high", "High"),
    "ARIMA": ("Low", "High", "Limited by four-to-five point regional histories"),
    "Random Forest": ("Moderate", "Moderate", "Limited by small training sample"),
    "XGBoost": ("High", "Moderate-low", "Limited by small training sample"),
    "Hybrid": ("High", "Moderate-low", "Weights collapse to ARIMA in supplied results"),
}
ranking["complexity"] = [qualitative[index][0] for index in ranking.index]
ranking["interpretability"] = [qualitative[index][1] for index in ranking.index]
ranking["robustness_note"] = [qualitative[index][2] for index in ranking.index]
ranking = ranking.sort_values(["mean_numeric_rank", "test_RMSE"]).reset_index().rename(
    columns={"model_name": "model_name", "index": "model_name"}
)
ranking.insert(
    0,
    "overall_rank",
    ranking["mean_numeric_rank"].rank(method="min").astype(int),
)
ranking.to_csv(FINAL_RESULTS / "final_model_ranking.csv", index=False, float_format="%.15g")
display(ranking)


## 6. Publication-quality figures


In [ ]:
test_plot = comparison.query("dataset_split == 'test'").copy()
metric_plot = test_plot.melt(id_vars="model_name", value_vars=["MAE", "RMSE", "MAPE", "R2"],
                             var_name="metric", value_name="value")
metric_plot["display_value"] = metric_plot["value"]
metric_plot.loc[metric_plot["metric"].isin(["MAE", "RMSE"]), "display_value"] /= 1e9
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, metric in zip(axes.flat, ["MAE", "RMSE", "MAPE", "R2"]):
    subset = metric_plot.query("metric == @metric").sort_values("display_value")
    sns.barplot(data=subset, x="display_value", y="model_name", color="#2A9D8F", ax=ax)
    ax.set_title(metric); ax.set_ylabel("")
    ax.set_xlabel("Billion kWh" if metric in ["MAE", "RMSE"] else metric)
fig.suptitle("Test Performance Across All Forecasting Approaches", fontsize=17, weight="bold")
fig.tight_layout(); fig.savefig(FINAL_FIGURES / "model_metric_comparison.png", bbox_inches="tight"); plt.show()

for metric, filename in [("RMSE", "RMSE_comparison.png"), ("MAE", "MAE_comparison.png")]:
    fig, ax = plt.subplots(figsize=(11, 6))
    subset = test_plot.sort_values(metric)
    sns.barplot(data=subset, x=subset[metric]/1e9, y="model_name", color="#1F4E78", ax=ax)
    ax.set_xlabel(f"Test {metric} (billion kWh)"); ax.set_ylabel("")
    ax.set_title(f"Test {metric} Comparison", weight="bold")
    fig.tight_layout(); fig.savefig(FINAL_FIGURES / filename, bbox_inches="tight"); plt.show()

test_predictions = all_predictions.query("dataset_split == 'test'")
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, (model_name, frame) in zip(axes.flat, test_predictions.groupby("model_name")):
    maximum = max(frame["actual"].max(), frame["predicted"].max()) / 1e9
    ax.scatter(frame["actual"]/1e9, frame["predicted"]/1e9, alpha=0.8)
    ax.plot([0, maximum], [0, maximum], color="black", linestyle="--")
    ax.set_title(model_name); ax.set_xlabel("Actual (billion kWh)"); ax.set_ylabel("Predicted")
fig.suptitle("Actual versus Predicted Consumption — 2022 Test", fontsize=17, weight="bold")
fig.tight_layout(); fig.savefig(FINAL_FIGURES / "actual_vs_predicted_all_models.png", bbox_inches="tight"); plt.show()

heat = test_predictions.pivot(index="region", columns="model_name", values="absolute_error") / 1e9
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax)
ax.set_title("Regional Absolute Error (billion kWh)", weight="bold")
fig.tight_layout(); fig.savefig(FINAL_FIGURES / "regional_error_heatmap.png", bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=test_predictions, x="absolute_error", y="model_name", ax=ax)
ax.set_xlabel("Absolute error (kWh)"); ax.set_ylabel("")
ax.set_title("Distribution of Regional Test Errors", weight="bold")
fig.tight_layout(); fig.savefig(FINAL_FIGURES / "model_error_distribution.png", bbox_inches="tight"); plt.show()

region_accuracy = test_predictions.groupby(["region", "model_name"], as_index=False)[
    "absolute_error"
].mean()
fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=region_accuracy, x="region", y=region_accuracy["absolute_error"]/1e9,
            hue="model_name", ax=ax)
ax.tick_params(axis="x", rotation=70); ax.set_ylabel("Absolute error (billion kWh)")
ax.set_title("Forecast Accuracy by Region", weight="bold")
fig.tight_layout(); fig.savefig(FINAL_FIGURES / "forecast_accuracy_by_region.png", bbox_inches="tight"); plt.show()

grouped = test_plot.copy()
grouped["approach_group"] = np.where(grouped["model_name"].str.contains("baseline"),
                                      "Baseline", "Advanced / hybrid")
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=grouped.sort_values("RMSE"), x=grouped["RMSE"]/1e9, y="model_name",
            hue="approach_group", dodge=False, ax=ax)
ax.set_xlabel("Test RMSE (billion kWh)"); ax.set_ylabel("")
ax.set_title("Baseline versus Advanced Forecasting Approaches", weight="bold")
fig.tight_layout(); fig.savefig(FINAL_FIGURES / "baseline_vs_advanced_models.png", bbox_inches="tight"); plt.show()


## 7. Scientific interpretation and thesis summaries


In [ ]:
first = ranking.iloc[0]
second = ranking.iloc[1]
rf_test = test_table.loc["Random Forest"]
moving_test = test_table.loc["Moving average baseline"]
arima_val = validation_table.loc["ARIMA"]
arima_test = test_table.loc["ARIMA"]

summary = f'''# Model Comparison Summary

## Scope

Eight approaches are compared using the frozen 2021 validation and 2022 test outputs.
Prediction-level regional analyses cover ARIMA, Random Forest, XGBoost, and Hybrid because
baseline prediction rows were not saved in Phase 3.

## Numerical findings

- **{first["model_name"]}** and **{second["model_name"]}** share the leading mean numerical
  rank ({first["mean_numeric_rank"]:.2f}). {first["model_name"]} is listed first only because
  its test RMSE is lower under the declared tie-break.
- Random Forest test RMSE: {rf_test["RMSE"]/1e9:.3f} billion kWh.
- Moving-average baseline test RMSE: {moving_test["RMSE"]/1e9:.3f} billion kWh.
- ARIMA validation RMSE: {arima_val["RMSE"]/1e9:.3f} billion kWh; test RMSE:
  {arima_test["RMSE"]/1e9:.3f} billion kWh.

## Interpretation

The leading numerical rank reflects several error criteria and validation-to-test stability,
not one metric. The moving-average baseline remains competitive and is much simpler. ARIMA is
strongest on validation but deteriorates on the single test year; supplied hybrid weights
collapse to ARIMA, so the hybrid does not diversify component errors.

## Statistical limitation

No formal significance test is performed. There are only 13 regional errors in one test year,
regions share national conditions, and only two holdout years exist. Standard independent-sample
or long-series forecast comparison tests would overstate evidence.
'''
(FINAL_RESULTS / "model_comparison_summary.md").write_text(summary, encoding="utf-8")

thesis = f'''# Thesis Results Summary

The 2022 evaluation does not support an unconditional claim that a single complex model is
universally best. **{first["model_name"]}** has the strongest aggregate multi-criterion
numerical rank, but performance is not uniformly dominant across validation, test metrics, and
regions. The moving-average baseline achieves competitive test errors with substantially lower
complexity, while ARIMA and the ARIMA-weighted hybrid show a validation-to-test reversal.

These findings must be interpreted in light of four model-ready target years, 13 regions,
regional heterogeneity, uncertain hybrid weights, and one final test year. Prediction accuracy
does not establish causal relationships between engineered variables and electricity demand.
Additional years and external validation are needed before operational deployment.
'''
(FINAL_RESULTS / "thesis_results_summary.md").write_text(thesis, encoding="utf-8")
print(summary)


## Conclusion

The evaluation preserves every poor or negative result and produces a transparent ranking.
Notebook 11 may use the highest-ranked prediction-eligible model for presentation, but must
retain the caveat that this choice is conditional on one small test year.
